## Paso 1 cargar las tablas de datos en Pandas:
En este paso, importamos los archivos CSV y Excel a DataFrames usando Pandas.

In [1]:
import pandas as pd

dimcal = pd.read_excel('DIM_CALENDAR (2).xlsx')
dimcat = pd.read_csv('DIM_CATEGORY (2).csv')
dp = pd.read_excel('DIM_PRODUCT (1).xlsx')
ds = pd.read_excel('DIM_SEGMENT (1).xlsx')
fs = pd.read_csv('FACT_SALES (1).csv')

## Paso 2 realizar la limpieza de datos:
- Corrigimos errores tipográficos y formatos incorrectos.
- Identificamos y manejamos valores nulos.
- Identificamos si hay duplicados y en caso de que los haya los eliminamos.

In [2]:
#Aqui corregimos errores tipograficos
dimcat['CATEGORY'] = dimcat['CATEGORY'].str.replace('FABRIC TREATMENT and SANIT\r\n', 'FABRIC TREATMENT & SANITATION')

#Manejamos valores nulos
dp = dp.fillna('')
ds = ds.fillna('')

#En la columna 'ITEM_DESCRIPTION' tambien se encuentra el item code de la columna 'ITEM' por lo que decidimos eliminarlo para que no se repita
#Para esto utilizamos la funcion str.rstrip y la lista de valores del 0-9 junto con las letras B y P porque los item code se conforman de estos
#Despues eliminamos los puntos y espacios
#Tambien utilizamos .str.lstrip para eliminar unos errores tipograficos en los cuales la descripcion empieza con 9
dp['ITEM_DESCRIPTION'] = dp['ITEM_DESCRIPTION'].str.rstrip('0123456789BP')
dp['ITEM_DESCRIPTION'] = dp['ITEM_DESCRIPTION'].str.rstrip('. ')
dp['ITEM_DESCRIPTION'] = dp['ITEM_DESCRIPTION'].str.lstrip('9')

#Estandarizamos a valores nulos
dp = dp.replace('NO DEFINIDO','')
ds = ds.replace('NO DEFINIDO','')

#Estandarizamos datos de fechas
dimcal['DATE'] = pd.to_datetime(dimcal['DATE'], format='%Y-%m-%d')

In [3]:
#Eliminamos los duplicados
dimcal = dimcal.drop_duplicates()
dp = dp.drop_duplicates()
ds = ds.drop_duplicates()
fs = fs.drop_duplicates()

## Paso 3 unir los DataFrames relevantes:
- Realizamos uniones entre los DataFrames para consolidar la información.

In [4]:
dp.rename(columns = {'ITEM':'ITEM_CODE', 'CATEGORY':'ID_CATEGORY'}, inplace=True)
ds.rename(columns = {'CATEGORY':'ID_CATEGORY'}, inplace=True)

#Unimos los dataframes en uno solo llamado 'df_completo' 
df1 = dp.merge(fs, how = 'inner')
df2 = dimcat.merge(ds, how = 'inner')
df_completo = df1.merge(df2, how ='inner')
df_completo = df_completo.merge(dimcal, how='inner')

#Eliminamos la columna 'WEEK' ya que existen otras dos columnas 'WEEK_NUMBER' y 'DATE' donde se encuentra la misma informacion
df_completo.drop(columns=['WEEK'])

,MANUFACTURER,BRAND,ITEM_CODE,ITEM_DESCRIPTION,ID_CATEGORY,FORMAT,ATTR1,ATTR2,ATTR3,TOTAL_UNIT_SALES,TOTAL_VALUE_SALES,TOTAL_UNIT_AVG_WEEKLY_SALES,REGION,CATEGORY,SEGMENT,YEAR,MONTH,WEEK_NUMBER,DATE
0,INDS. ALEN,CLORALEX,0000075000592,CLORALEX EL RENDIDOR BOT.PLAST. 250ML NAL,1,LIQUIDO,CLORO,CLORO,,0.034,0.142,8.500,TOTAL AUTOS SCANNING MEXICO,FABRIC TREATMENT & SANITATION,BLEACH,2022,11,45,2022-11-13
1,INDS. ALEN,CLORALEX,0000075000592,CLORALEX EL RENDIDOR BOT.PLAST. 250ML NAL,1,LIQUIDO,CLORO,CLORO,,0.002,0.009,1.000,TOTAL AUTOS SCANNING MEXICO,FABRIC TREATMENT & SANITATION,BLEACH,2022,12,48,2022-12-04
2,INDS. ALEN,CLORALEX,0000075000592,CLORALEX EL RENDIDOR BOT.PLAST. 250ML NAL,1,LIQUIDO,CLORO,CLORO,,0.432,1.956,14.400,TOTAL AUTOS SCANNING MEXICO,FABRIC TREATMENT & SANITATION,BLEACH,2022,1,3,2022-01-23
3,INDS. ALEN,CLORALEX,0000075000592,CLORALEX EL RENDIDOR BOT.PLAST. 250ML NAL,1,LIQUIDO,CLORO,CLORO,,0.233,0.779,11.095,TOTAL AUTOS SCANNING MEXICO,FABRIC TREATMENT & SANITATION,BLEACH,2022,2,5,2022-02-06
4,INDS. ALEN,CLORALEX,0000075000592,CLORALEX EL RENDIDOR BOT.PLAST. 250ML NAL,1,LIQUIDO,CLORO,CLORO,,0.055,0.251,5.500,TOTAL AUTOS SCANNING MEXICO,FABRIC TREATMENT & SANITATION,BLEACH,2022,3,9,2022-03-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121997,CLOROX,CLOROX,7502240948188BP1,CLOROX BLANQ930M+MMBLANQINOD709M+MMFIBRAULTRA1...,1,LIQUIDO,CLORO,CLORO,,0.013,0.692,1.300,TOTAL AUTOS AREA 2,FABRIC TREATMENT & SANITATION,BLEACH,2023,7,26,2023-07-03
121998,CLOROX,CLOROX,7502240948188BP1,CLOROX BLANQ930M+MMBLANQINOD709M+MMFIBRAULTRA1...,1,LIQUIDO,CLORO,CLORO,,0.010,0.548,1.667,TOTAL AUTOS SCANNING MEXICO,FABRIC TREATMENT & SANITATION,BLEACH,2023,7,28,2023-07-17
121999,CLOROX,CLOROX,7502240948188BP1,CLOROX BLANQ930M+MMBLANQINOD709M+MMFIBRAULTRA1...,1,LIQUIDO,CLORO,CLORO,,0.013,0.692,1.300,TOTAL AUTOS SCANNING MEXICO,FABRIC TREATMENT & SANITATION,BLEACH,2023,7,26,2023-07-03
122000,CLOROX,CLOROX,7502240948188BP1,CLOROX BLANQ930M+MMBLANQINOD709M+MMFIBRAULTRA1...,1,LIQUIDO,CLORO,CLORO,,0.013,0.720,1.083,TOTAL AUTOS AREA 2,FABRIC TREATMENT & SANITATION,BLEACH,2023,6,25,2023-06-26


## Paso 4 aplicar transformaciones necesarias:
- Creamos una nueva columna calculada del precio unitario

In [5]:
df_completo['UNIT_PRICE'] =df_completo['TOTAL_VALUE_SALES'] / df_completo['TOTAL_UNIT_SALES']
df_completo.head()

,MANUFACTURER,BRAND,ITEM_CODE,ITEM_DESCRIPTION,ID_CATEGORY,FORMAT,ATTR1,ATTR2,ATTR3,WEEK,...,TOTAL_VALUE_SALES,TOTAL_UNIT_AVG_WEEKLY_SALES,REGION,CATEGORY,SEGMENT,YEAR,MONTH,WEEK_NUMBER,DATE,UNIT_PRICE
0,INDS. ALEN,CLORALEX,0000075000592,CLORALEX EL RENDIDOR BOT.PLAST. 250ML NAL,1,LIQUIDO,CLORO,CLORO,,45-22,...,0.142,8.500,TOTAL AUTOS SCANNING MEXICO,FABRIC TREATMENT & SANITATION,BLEACH,2022,11,45,2022-11-13,4.176471
1,INDS. ALEN,CLORALEX,0000075000592,CLORALEX EL RENDIDOR BOT.PLAST. 250ML NAL,1,LIQUIDO,CLORO,CLORO,,48-22,...,0.009,1.000,TOTAL AUTOS SCANNING MEXICO,FABRIC TREATMENT & SANITATION,BLEACH,2022,12,48,2022-12-04,4.500000
2,INDS. ALEN,CLORALEX,0000075000592,CLORALEX EL RENDIDOR BOT.PLAST. 250ML NAL,1,LIQUIDO,CLORO,CLORO,,03-22,...,1.956,14.400,TOTAL AUTOS SCANNING MEXICO,FABRIC TREATMENT & SANITATION,BLEACH,2022,1,3,2022-01-23,4.527778
3,INDS. ALEN,CLORALEX,0000075000592,CLORALEX EL RENDIDOR BOT.PLAST. 250ML NAL,1,LIQUIDO,CLORO,CLORO,,05-22,...,0.779,11.095,TOTAL AUTOS SCANNING MEXICO,FABRIC TREATMENT & SANITATION,BLEACH,2022,2,5,2022-02-06,3.343348
4,INDS. ALEN,CLORALEX,0000075000592,CLORALEX EL RENDIDOR BOT.PLAST. 250ML NAL,1,LIQUIDO,CLORO,CLORO,,09-22,...,0.251,5.500,TOTAL AUTOS SCANNING MEXICO,FABRIC TREATMENT & SANITATION,BLEACH,2022,3,9,2022-03-06,4.563636


## Paso 5 guardar el conjunto de datos consolidado:
- Guardamos el DataFrame consolidado en un nuevo archivo CSV

In [6]:
df_completo.to_csv('Entregable1.csv', index=False)